<a href="https://colab.research.google.com/github/anarghya130607-hash/Whatsapp_chat_analyzer/blob/main/Whatsapp_message_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import numpy as np
from datetime import datetime

dataset = "/content/DADS Minor PROJECT dataset.txt"
group_name = "Hostel bois 4ever"

# Words to ignore when counting "favourite words"
neglected_words = {
    "i", "is", "the", "a", "an", "and", "or", "to", "of", "in", "on", "for",
    "you", "it", "its", "this", "that", "these", "those", "hai", "ke", "ka",
    "ki", "ho", "na", "hi", "me", "my", "your", "we", "are", "with", "so",
    "at", "be", "been", "being", "not", "am", "was", "were", "he", "she",
    "his", "her", "him", "they", "them", "their", "our", "us", "as", "by",
    "but", "if", "then", "than", "so", "just", "up", "out", "about", "into",
    "over", "after", "before", "again", "here", "there", "when", "where",
    "why", "how", "all", "any", "both", "each", "few", "more", "most",
    "other", "some", "such", "no", "nor", "only", "own", "same", "too",
    "very", "one", "from", "have", "has", "had", "do", "does", "did",
    "will", "would", "shall", "should", "may", "might", "must", "can",
    "could", "which", "who", "whom", "what", "yourself", "himself",
    "herself", "themselves", "myself", "ourselves", "i'm", "it's",
    "don't", "didn't", "can't", "won't", "started", "get", "got"
}

Top_words = 10   # to show top group words
Top_words_per_person = 3    # to show how many top words per person

#Archetype detection thresholds
Night_owl_start = 23     # night window start (23 to 11 PM)
Night_owl_end = 5      #  night window end, exclusive (5 to up to 4:59 AM)
Night_owl_threshold = 0.60  # 60%+ of messages in that window = night owl signal

Spammer_burst_threshold = 3.0   # avg consecutive same-person messages to flag as "spammer"
Storyteller_threshold = 30   # avg words/message to flag as "storyteller"
Drama_queen_Threshold = 0.30 # 30%+ messages to flag as "drama queen"
Ghost_silent_threshold = 0.60 # silent on 60%+ of days to flag as "ghost"
Questioner_threshold = 0.25 # 25%+ messages ending in '?' to flag as "question master"

Sympathetic_words = ["okay", "safe", "eat", "sleep", "take care", "are you", "please",
    "reminder", "drink water", "don't forget", "reached", "call home",
    "worried", "stress"]
Humor_words = ["lol", "lmao", "haha", "rofl", "lmfao"]
Excitment_words = ["scene", "bhai", "chal", "aaja", "let's go", "lit", "fire"]

#  FEATURE 1 — THE CHAT PARSER
def looks_like_date_start(line):
    if len(line) < 8:
        return False
    d = line[0:2] + line[3:5] + line[6:8]
    return (line[2] == "/" and line[5] == "/" and d.isdigit())

def parse_chat(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_lines = f.read().split("\n")

    messages = []
    system_msg_count = 0
    media_omitted_count = 0
    deleted_msg_count = 0

    pending = None  # holds the message currently being built (for multi-line)

    for line in raw_lines:
        if line.strip() == "":
            continue  # to skip empty lines

        if not looks_like_date_start(line):
            if pending is not None:
                pending["text"] += " " + line.strip()
            continue
        if pending is not None:
            messages.append(pending)
            pending = None

        parts = line.split(" - ", 1)
        if len(parts)!= 2:
            system_msg_count += 1
            continue
        timestamp_str, rest = parts

        # try to split "Sender: message"
        dialogue_split = rest.split(": ", 1)
        if len(dialogue_split) != 2:
            system_msg_count += 1
            continue

        sender, text = dialogue_split
        sender = sender.strip()

        try:
            dt = datetime.strptime(timestamp_str.strip(), "%d/%m/%y, %H:%M")
        except ValueError:
            system_msg_count += 1
            continue

        if text == "This message was deleted":
            deleted_msg_count += 1
            kind = "deleted"
        elif text == "<Media omitted>":
            media_omitted_count += 1
            kind = "media"
        else:
            kind = "text"

        pending = {"timestamp": timestamp_str.strip(), "dt": dt,
                   "sender": sender, "text": text, "kind": kind}

    if pending is not None:
        messages.append(pending)

    return messages, system_msg_count, media_omitted_count, deleted_msg_count


messages, system_msg_count, media_omitted_count, deleted_msg_count = parse_chat(dataset)

participants = sorted(set(m["sender"] for m in messages))
print(f"Successfully parsed {len(messages)} messages from {len(participants)} "
      f"participants, skipped {system_msg_count} system messages, {media_omitted_count} "
      f"media-omitted, {deleted_msg_count} deleted messages.")


#  FEATURE 2 — GROUP OVERVIEW
person_counts = {}
for m in messages:
    person_counts[m["sender"]] = person_counts.get(m["sender"], 0) + 1

total_messages = len(messages)
first_day = min(m["dt"] for m in messages).date()
last_day = max(m["dt"] for m in messages).date()
total_days = (last_day - first_day).days + 1

ranked_people = sorted(person_counts.items(), key=lambda kv: kv[1], reverse=True)

#  FEATURE 3 — MOST ACTIVE DAY & HOUR
day_count = {}
hour_count = {}
for m in messages:
    days = m["dt"].date()
    hours = m["dt"].hour
    day_count[days] = day_count.get(days, 0) + 1
    hour_count[hours] = hour_count.get(hours, 0) + 1

busiest_day, busiest_day_count = max(day_count.items(), key=lambda kv: kv[1])
busiest_hour, busiest_hour_count = max(hour_count.items(), key=lambda kv: kv[1])

#  FEATURE 4 — NUMPY ACTIVITY HEATMAP
person_index = {p: i for i, p in enumerate(participants)}
heatmap = np.zeros((len(participants), 24), dtype=int)

for m in messages:
    row = person_index[m["sender"]]
    column = m["dt"].hour
    heatmap[row, column] += 1

def render_heatmap(matrix, people):
    shades = [(0.25, ".  "), (0.50, "\u2591 "), (0.75, "\u2592 "), (1.01, "\u2588 ")]
    lines = []
    header = "        " + "".join(f"{h:02d} " for h in range(0, 24, 3))
    lines.append(header)
    for i, person in enumerate(people):
        row = matrix[i]
        row_max = row.max() if row.max() > 0 else 1
        cells = []
        for h in range(0, 24, 3):
            val = row[h:h + 3].sum() / 3.0  # average over the 3-hour block
            ratio = val / row_max
            for limit, pattern in shades:
                if ratio <= limit:
                    cells.append(pattern)
                    break
        lines.append(f"{person:<8}" + "".join(cells))
    return "\n".join(lines)

#  FEATURE 5 — TOP WORDS
Punctuation = "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"

def tokenize(text):
    words = []
    for raw in text.split(" "):
        w = raw.lower().strip(Punctuation)
        if w and w not in neglected_words:
            words.append(w)
    return words

word_count= {}
person_word_count = {p: {} for p in participants}
for m in messages:
    if m["kind"] != "text":
        continue
    for w in tokenize(m["text"]):
        word_count[w] = word_count.get(w, 0) + 1
        person_word_count[m["sender"]][w] = person_word_count[m["sender"]].get(w, 0) + 1

top_words = sorted(word_count.items(), key=lambda kv: kv[1], reverse=True)[:Top_words]

#  FEATURE 6 — RESPONSE SPEED & SILENT STREAKS
sorted_msgs = sorted(messages, key=lambda m: m["dt"])

response_gaps = {p: [] for p in participants}
for i in range(1, len(sorted_msgs)):
    prev_m, cur_m = sorted_msgs[i - 1], sorted_msgs[i]
    if cur_m["sender"] != prev_m["sender"]:
        gap_seconds = (cur_m["dt"] - prev_m["dt"]).total_seconds()
        response_gaps[cur_m["sender"]].append(gap_seconds)

avg_response = {}
for p in participants:
    gaps = response_gaps[p]
    avg_response[p] = (sum(gaps) / len(gaps)) if gaps else None

active_days_by_person = {p: set() for p in participants}
for m in messages:
    active_days_by_person[m["sender"]].add(m["dt"].date())

all_dates = [first_day]
d = first_day
while d < last_day:
    d_next = d.fromordinal(d.toordinal() + 1)
    all_dates.append(d_next)
    d = d_next

silent_streaks = {}
for p in participants:
    active = active_days_by_person[p]
    longest, longest_start, longest_end = 0, None, None
    current_len, current_start = 0, None
    for day in all_dates:
        if day not in active:
            if current_len == 0:
                current_start = day
            current_len += 1
            if current_len > longest:
                longest, longest_start, longest_end = current_len, current_start, day
        else:
            cur_len = 0
    silent_streaks[p] = (longest, longest_start, longest_end)

#  FEATURE 7 — PERSONALITY ARCHETYPE DETECTION
def burst_score(person):
    runs, current = [], 0
    for m in sorted_msgs:
        if m["sender"] == person:
            current += 1
        else:
            if current:
                runs.append(current)
            current = 0
    if current:
        runs.append(current)
    return (sum(runs) / len(runs)) if runs else 0.0


def caring_score(person):
    total = 0
    for m in messages:
        if m["sender"] != person or m["kind"] != "text":
            continue
        low = m["text"].lower()
        total += sum(low.count(k) for k in Sympathetic_words)
    return total


def night_owl_pct(person):
    row = heatmap[person_index[person]]
    total = row.sum()
    if total == 0:
        return 0.0
    if Night_owl_start > Night_owl_end:
        night = row[Night_owl_start:].sum() + row[:Night_owl_end].sum()
    else:
        night = row[Night_owl_start:Night_owl_end].sum()
    return night / total


def avg_words(person):
    texts = [m["text"] for m in messages if m["sender"] == person and m["kind"] == "text"]
    if not texts:
        return 0.0
    return sum(len(t.split()) for t in texts) / len(texts)


def caps_pct(person):
    texts = [m["text"] for m in messages if m["sender"] == person and m["kind"] == "text"]
    if not texts:
        return 0.0
    flagged = 0
    for t in texts:
        letters_only = "".join(c for c in t if c.isalpha())
        is_caps = len(letters_only) >= 3 and letters_only.isupper()
        if is_caps or t.count("!") >= 2:
            flagged += 1
    return flagged / len(texts)


def silent_pct(person):
    return silent_streaks[person][0] / total_days if total_days else 0.0
    # (using longest streak; days-silent-total variant also common — see README notes)


def comedian_pct(person):
    texts = [m["text"].lower() for m in messages if m["sender"] == person and m["kind"] == "text"]
    if not texts:
        return 0.0
    hits = sum(1 for t in texts if any(k in t for k in Humor_words))
    return hits / len(texts)


def question_pct(person):
    texts = [m["text"] for m in messages if m["sender"] == person and m["kind"] == "text"]
    if not texts:
        return 0.0
    hits = sum(1 for t in texts if t.strip().endswith("?"))
    return hits / len(texts)


def hype_pct(person):
    texts = [m["text"].lower() for m in messages if m["sender"] == person and m["kind"] == "text"]
    if not texts:
        return 0.0
    hits = sum(1 for t in texts if any(k in t for k in Excitment_words))
    return hits / len(texts)


ARCHETYPES = [
    ("THE SPAMMER",         lambda p: 1.0 if burst_score(p) > Spammer_burst_threshold else 0.0,
                            lambda p: f"avg {burst_score(p):.1f} msgs in a row"),
    ("THE GROUP MOM",       lambda p: caring_score(p),
                            lambda p: f"caring keyword score: {caring_score(p)}"),
    ("THE NIGHT OWL",       lambda p: night_owl_pct(p) if night_owl_pct(p) > Night_owl_threshold else 0.0,
                            lambda p: f"{night_owl_pct(p)*100:.1f}% msgs between 23h-04h"),
    ("THE STORYTELLER",     lambda p: avg_words(p) if avg_words(p) > Storyteller_threshold else 0.0,
                            lambda p: f"avg {avg_words(p):.1f} words per msg"),
    ("THE DRAMA QUEEN",     lambda p: caps_pct(p) if caps_pct(p) > Drama_queen_Threshold else 0.0,
                            lambda p: f"{caps_pct(p)*100:.1f}% ALL-CAPS messages"),
    ("THE GHOST",           lambda p: silent_pct(p) if silent_pct(p) > Ghost_silent_threshold else 0.0,
                            lambda p: f"silent streak covers {silent_pct(p)*100:.0f}% of the period"),
    ("THE COMEDIAN",        lambda p: comedian_pct(p),
                            lambda p: f"{comedian_pct(p)*100:.1f}% messages have lol/haha/lmao"),
    ("THE QUESTION MASTER", lambda p: question_pct(p) if question_pct(p) > Questioner_threshold else 0.0,
                            lambda p: f"{question_pct(p)*100:.1f}% messages end in '?'"),
    ("THE HYPE BEAST",      lambda p: hype_pct(p),   # <-- bonus 9th archetype (invented)
                            lambda p: f"{hype_pct(p)*100:.1f}% messages use hype words (bhai/scene/chal...)"),
]

archetype = {}
for p in participants:
    scores = [(name, score_fn(p), detail_fn(p)) for name, score_fn, detail_fn in ARCHETYPES]
    scores.sort(key=lambda t: t[1], reverse=True)
    archetype[p] = scores[0]  # (name, score, detail_string)


#  FEATURE 8 — THE FINAL REPORT
def bar(count, max_count, width=20):
    filled = int((count / max_count) * width) if max_count else 0
    return "\u2588" * filled if filled > 0 else "."


def print_report():
    W = 60
    print("=" * W)
    print(f" GROUPDNA REPORT \u2014 \"{group_name}\"")
    print(f" {total_days} days \u2022 {total_messages:,} messages \u2022 {len(participants)} members")
    print("=" * W)
    print(f" Period       : {first_day.strftime('%d %B %Y')} to {last_day.strftime('%d %B %Y')}")
    print(f" Busiest day  : {busiest_day.strftime('%d %B %Y')} ({busiest_day_count} messages)")
    print(f" Busiest hour : {busiest_hour:02d}:00 - {(busiest_hour+1)%24:02d}:00")
    print()
    print(" MESSAGES PER PERSON")
    max_c = ranked_people[0][1]
    for person, count in ranked_people:
        pct = count / total_messages * 100
        print(f" {person:<8}{bar(count, max_c):<22}{count:>5} ({pct:4.1f}%)")
    print()
    print(" ACTIVITY HEATMAP (hour of day, 3-hour blocks, 00 to 23)")
    print(render_heatmap(heatmap, participants))
    print()
    print(f" THIS GROUP'S FAVOURITE WORDS (top {Top_words})")
    max_w = top_words[0][1] if top_words else 1
    for word, count in top_words:
        print(f" {word:<10}{bar(count, max_w):<22}{count}")
    print()
    print(" RESPONSE PATTERNS")
    valid_resp = {p: v for p, v in avg_response.items() if v is not None}
    if valid_resp:
        fastest = min(valid_resp.items(), key=lambda kv: kv[1])
        slowest = max(valid_resp.items(), key=lambda kv: kv[1])
        print(f" Fastest replier : {fastest[0]} (avg {fastest[1]/60:.1f} minutes)")
        print(f" Slowest replier : {slowest[0]} (avg {slowest[1]/3600:.1f} hours)")
    print()
    print(" LONGEST SILENT STREAKS")
    for person, (streak_len, s_start, s_end) in sorted(silent_streaks.items(), key=lambda kv: kv[1][0], reverse=True):
        if streak_len > 0:
            print(f" {person:<8}: {streak_len} days ({s_start.strftime('%d %b')} - {s_end.strftime('%d %b')})")
        else:
            print(f" {person:<8}: 0 days (never went silent)")
    print()
    print(" PERSONALITY ARCHETYPES")
    for person in participants:
        name, score, detail = archetype[person]
        print(f" {person:<8}\u2192 {name} ({detail})")
    print()


print_report()

Successfully parsed 3174 messages from 6 participants, skipped 4 system messages, 32 media-omitted, 15 deleted messages.
 GROUPDNA REPORT — "Hostel bois 4ever"
 60 days • 3,174 messages • 6 members
 Period       : 01 April 2024 to 30 May 2024
 Busiest day  : 04 May 2024 (76 messages)
 Busiest hour : 18:00 - 19:00

 MESSAGES PER PERSON
 Rahul   ████████████████████    953 (30.0%)
 Priya   ███████████████         718 (22.6%)
 Neha    █████████████           635 (20.0%)
 Aman    ██████████              490 (15.4%)
 Karan   ███████                 354 (11.2%)
 Vikas   .                        24 ( 0.8%)

 ACTIVITY HEATMAP (hour of day, 3-hour blocks, 00 to 23)
        00 03 06 09 12 15 18 21 
Aman    ▒ ▒ .  .  .  .  .  ░ 
Karan   .  .  .  ░ █ ▒ ▒ ░ 
Neha    .  .  ░ ▒ ▒ ▒ █ ░ 
Priya   .  .  ░ █ █ ▒ ▒ ░ 
Rahul   .  .  .  .  ░ ▒ ▒ ▒ 
Vikas   .  .  ░ .  ░ ▒ ▒ ░ 

 THIS GROUP'S FAVOURITE WORDS (top 10)
 guys      ████████████████████  318
 today     ████████████████      257
 everyone  ████████